# Understanding and mapping the Global Human Settlement Layer

1. Extract settlement extent from [UCDB](https://human-settlement.emergency.copernicus.eu/ghs_ucdb_2024.php)  
2. Clip out ghsl layers for selected area  
3. Convert ghsl layers to "year of built" layer (GOSTrocks.ghslmisc function) 
4. ...  
5. Profit  


In [ ]:
import sys, os
import rasterio

import geopandas as gpd
import pandas as pd

sys.path.insert(0, "../src")

import GOSTrocks.ghslMisc as ghslMisc
import GOSTrocks.rasterMisc as rMisc
import GOSTrocks.mapMisc as mapMisc

%load_ext autoreload
%autoreload 2

In [ ]:
ucdb_path = r'C:\WBG\Work\data\URBAN\GHS_UCDB_GLOBE_R2024A.gpkg'
ghsl_folder = r'C:\WBG\Work\data\GHSL\BUILT'
ghsl_files = [os.path.join(ghsl_folder, f) for f in os.listdir(ghsl_folder) if f.endswith('.tif')]
out_folder = r'C:\WBG\Work\Projects\SDG_ATLAS\Urban'
if not os.path.exists(out_folder):
    os.makedirs(out_folder)

# gpd.list_layers(ucdb_path)
in_ucdb = gpd.read_file(ucdb_path, layer="GHS_UCDB_THEME_GENERAL_CHARACTERISTICS_GLOBE_R2024A")
in_ucdb.head()


In [ ]:
#in_ucdb.loc[in_ucdb['GC_CNT_UNN_2025'] == "South Sudan"].to_file("C:/WBG/Work/Projects/SSD_Health/DATA/UCDB_cities.gpkg", layer="South_Sudan_cities", driver="GPKG")

In [ ]:
cities = ['Quebec', "Ouagadougou", "Toronto", "Kumasi", "Yaoundé", "Addis Ababa", "Dhaka", "Bucharest"]
for cur_city in cities:
    temp_out_folder = os.path.join(out_folder, cur_city)
    if not os.path.exists(temp_out_folder):
        os.makedirs(temp_out_folder)
    cur_city_ucdb = in_ucdb.loc[in_ucdb['GC_UCN_MAI_2025'] == cur_city].copy()
    if cur_city_ucdb.empty:
        print(f"City {cur_city} not found in UCDB. Skipping.")
        continue
    else:
        combined_ghsl_file = os.path.join(temp_out_folder, f"{cur_city}_combined_ghsl.tif")
        map_file = os.path.join(temp_out_folder, f"{cur_city}_map.png")    
        out_ghsl_files = []
        for ghsl_file in ghsl_files:
            cur_out_file = os.path.join(temp_out_folder, os.path.basename(ghsl_file))
            out_ghsl_files.append(cur_out_file)
            with rasterio.open(ghsl_file) as src:
                rMisc.clipRaster(src, cur_city_ucdb, cur_out_file, crop=False)

        ghslMisc.combine_ghsl_annual(out_ghsl_files, out_file=combined_ghsl_file, built_thresh=1000)
        plt, fig, ax = mapMisc.static_map_raster(rasterio.open(combined_ghsl_file), thresh=list(range(1975, 2026, 5)),
                                                scale_bar={'dx': 5, 'location': 'lower right'})
        ax.set_title(cur_city)
        ax.set_axis_off()
        plt.savefig(map_file, bbox_inches='tight', dpi=300)

    
